# HW3: LLM-as-Judge Eval

In this homework you'll create an LLM judge to evaluate whether your recipe agent respects dietary restrictions.

**What you'll learn:**
- How to upload a CSV as a LangSmith dataset
- How to create dataset splits
- How to define an LLM-as-judge evaluator in the LangSmith UI
- How to run experiments with `evaluate()` and compare results

## Setup

In [ ]:
import sys
sys.path.insert(0, "../hw1")

from dotenv import load_dotenv

load_dotenv()

## 1. Upload Dataset (UI)

Upload `data.csv` as a LangSmith dataset.

### Steps:
1. Go to [smith.langchain.com](https://smith.langchain.com)
2. Navigate to **Datasets & Testing** in the left sidebar
3. Click **New Dataset** > **Upload CSV**
4. Select `hw3/data.csv`
5. Map the columns:
   - `input` → Input
   - `output` → Output (this is the pre-generated agent response)
   - `expected` → Output (or use as a reference field)
6. Name the dataset (e.g., `recipe-dietary-compliance`)

Alternatively, upload via the SDK:

In [ ]:
import pandas as pd
from langsmith import Client

client = Client()

df = pd.read_csv("data.csv")

dataset = client.create_dataset(
    dataset_name="recipe-dietary-compliance",
    description="Recipe agent responses evaluated for dietary restriction compliance.",
)

for _, row in df.iterrows():
    client.create_example(
        dataset_id=dataset.id,
        inputs={"input": row["input"]},
        outputs={"expected": row["expected"]},
    )

print(f"Created dataset '{dataset.name}' with {len(df)} examples.")

## 2. Create a Dataset Split (UI)

Splits let you organize examples into subsets (e.g., `train` / `test`).

### Steps:
1. Open the dataset you just created
2. Select the examples you want in the split
3. Click **Add to Split** and create a new split (e.g., `test`)

For this homework, you can put all examples in a single `test` split.

## 3. Create an LLM-as-Judge Evaluator (UI)

Define a judge that evaluates dietary restriction compliance.

### Steps:
1. Open your dataset in LangSmith
2. Click **Add Evaluator** (or go to the Evaluators tab)
3. Choose **LLM-as-Judge**
4. Configure the judge:
   - **Name:** `dietary-compliance`
   - **Prompt:** Write a prompt that instructs the LLM to check whether the recipe output complies with the stated dietary restriction. For example:
     ```
     You are evaluating whether a recipe suggestion correctly follows the user's dietary restrictions.
     
     User request: {input}
     Agent response: {output}
     Expected criteria: {expected}
     
     Does the recipe fully comply with the dietary restriction? 
     Score 1 if compliant, 0 if not. Explain your reasoning.
     ```
   - **Score schema:** Binary (0 or 1) or continuous (0.0 to 1.0)
5. Save the evaluator — it's now linked to this dataset

> **Key point:** Because the evaluator is linked to the dataset, you do NOT need to pass it to `evaluate()`. It will run automatically when experiments are run against this dataset.

## 4. Run Experiments

Run two experiments with different agent configurations. The UI-defined judge will evaluate both automatically.

In [ ]:
from agent import get_agent
from langchain.messages import HumanMessage
from langsmith import evaluate

### Experiment 1: Default agent (gpt-4o-mini)

In [ ]:
agent_mini = get_agent()


def run_agent_mini(inputs: dict) -> dict:
    response = agent_mini.invoke(
        {"messages": [HumanMessage(content=inputs["input"])]}
    )
    return {"output": response["messages"][-1].content}


results_mini = evaluate(
    run_agent_mini,
    data="recipe-dietary-compliance",
    experiment_prefix="recipe-gpt4o-mini",
)

### Experiment 2: Stronger model (gpt-4o)

Let's see if a more capable model does better at respecting dietary restrictions.

In [ ]:
from langchain.agents import create_agent
from agent import web_search, SYSTEM_PROMPT

agent_4o = create_agent(
    model="gpt-4o",
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
)


def run_agent_4o(inputs: dict) -> dict:
    response = agent_4o.invoke(
        {"messages": [HumanMessage(content=inputs["input"])]}
    )
    return {"output": response["messages"][-1].content}


results_4o = evaluate(
    run_agent_4o,
    data="recipe-dietary-compliance",
    experiment_prefix="recipe-gpt4o",
)

## 5. Compare Experiments (UI)

### Steps:
1. Go to your dataset in LangSmith
2. Click the **Experiments** tab
3. You should see both experiments (`recipe-gpt4o-mini-*` and `recipe-gpt4o-*`)
4. Select both and click **Compare**
5. Review:
   - Side-by-side outputs for each example
   - Judge scores for dietary compliance
   - Which model handles restrictions better

### Aligning Results:
- Click on individual examples to see the judge's reasoning
- If you disagree with the judge, you can add manual annotations to refine the evaluation
- Use this to iterate on your judge prompt if needed